# Material-weighted energy estimation

The physical energy norm requires inverse-permeability weighting of flux defects and a certified lower ellipticity bound in Poincare terms. Mixed boundary data must be exactly represented.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/2376f3aba97d397677e6a22e47467976150a2eafc05be8022c6f5b8df3e89710/31_weighted_energy_estimator-companion.zip"
COMPANION_SHA256 = "2376f3aba97d397677e6a22e47467976150a2eafc05be8022c6f5b8df3e89710"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/31_weighted_energy_estimator.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

from pymhm import TriangleMesh
from pymhm.estimators.darcy_energy import estimate_weighted_darcy_error

solution_definition = define_darcy(
    TriangleMesh.unit_square(),
    degree=2,
    local_refinement=2,
    permeability=2.0,
    dirichlet=lambda x: x[:, 0],
)
solution_system = assemble(solution_definition.problem)
solution = recover_darcy(
    solution_definition,
    solution_system,
    solution_system.solve(constraints=pressure_constraints(solution_definition, solution_system)),
)
estimator = estimate_weighted_darcy_error(solution, dirichlet=lambda x: x[:, 0])
assert estimator.total < 1e-11
estimator.total


The full research campaign is acquired separately from this lightweight notebook. The following cells display its numerical record and publication figures.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    from pathlib import Path
    import json
    from IPython.display import Image, display
    record = json.loads((root / "examples/results/weighted-estimator.json").read_text())
    record


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/weighted-estimator/convergence.png")))
